# CS 195: Natural Language Processing
## Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F4_1_Embeddings.ipynb)


## References

[SLP: Embeddings, Chapter 5 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

[Word2Vec Tutorial - The Skip-Gram Model by Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)

[Word2Vec - Negative Sampling made easy by Munesh Lakhey](https://medium.com/@mnshonco/word2vec-negative-sampling-made-easy-9a587cb4695f)

[PyTorch `nn.Embedding` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

[PyTorch `torch.nn.functional.one_hot` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)


In [1]:
#import sys
#!{sys.executable} -m pip install datasets torch scikit-learn transformers tokenizers


In [2]:
# lets set random seeds for reproducibility
import random
import torch

random.seed(42)
torch.manual_seed(42)

## Dataset for today

AG News dataset
* short news articles
* four classes: World, Sports, Business, Sci/Tech

https://huggingface.co/datasets/fancyzhx/ag_news


In [3]:
from datasets import load_dataset
data = load_dataset("ag_news")

print(data["train"]["text"][0])

# 0 is World
# 1 is Sports
# 2 is Business
# 3 is Sci/Tech
print(data["train"]["label"][0])


Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2


## Multiclass classification example

This should look very familiar from last class.
We'll reuse the same multiclass PyTorch pattern (`CrossEntropyLoss` + `argmax`) on AG News.


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)

model = nn.Sequential(
    nn.Linear(5000, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

for epoch in range(200):
    optimizer.zero_grad()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")

with torch.no_grad():
    logits = model(X_test)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())


Epoch 1, loss = 1.3862
Epoch 21, loss = 0.1606
Epoch 41, loss = 0.0603
Epoch 61, loss = 0.0156
Epoch 81, loss = 0.0057
Epoch 101, loss = 0.0032
Epoch 121, loss = 0.0024
Epoch 141, loss = 0.0019
Epoch 161, loss = 0.0016
Epoch 181, loss = 0.0014
Accuracy: 0.8930262923240662


## Why move on from this?

Just using the integers: token ids are arbitary - no meaning attached to any given number

Bag of Words/TF-IDF: similar texts might have similar encodings, but nothing about word order is captured

We want both! We want encodings that take into account word order and where similar texts will have similar numerical representation

For this, we need **embeddings**


## Word Embeddings

Don't just represent words with a single number - use a whole vector

<div>
   <img src="images/embeddings.png"> 
</div>

image source: https://stackoverflow.com/questions/46155868/keras-embedding-layer

in reality, you can't label the dimensions with exact meanings like "living being", "feline", etc.
* but you don't need to!

## How do we come up with these embeddings?

First, we'll come up with a "fake" learning problem and then extract information about words from the model

**fake learning problem** skip-grams: predict the words that appear around a given word

We can pick any window size - this one uses size 2 (2 words before and 2 after)


<div>
   <img src="images/skip_gram_problem.png"> 
</div>

image source: http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/

## One Hot Encoding

Before we can create a model for this, we need an initial numerical encoding of words

**One Hot Encoding** uses a vector with length equal to the size of the vocabulary
* 1 in the spot representing that word
* 0 in all others

|            | the | quick | brown | fox | jumps | over | lazy | dog |
|------------|-----|-----|-----|----|-----|-----|-----|-----|
| "fox" | 0   | 0   | 0   | 1  | 0   | 0   | 0   | 0   |
| "dog" | 0 | 0   | 0   | 0   | 0  | 0   | 0   | 1   |

We can use PyTorch `torch.nn.functional.one_hot` for this


In [5]:
import torch
import torch.nn.functional as F

# put a 1 at index 3 of 8
print(F.one_hot(torch.tensor(3), num_classes=8).float())


tensor([0., 0., 0., 1., 0., 0., 0., 0.])


## Let's get started
Here are some toy sentences we can work with.


In [6]:
sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

We could use a pre-trained tokenizer like we've done before, but that would have too big of a vocabulary for this demo. Let's train a new one on our toy sentences.

In [7]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

vocabulary_size = 200

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

tokenizer.train_from_iterator(sentences, trainer)

example_encoding = tokenizer.encode(sentences[0])
print(example_encoding.tokens)
print(example_encoding.ids)

vocabulary_size = tokenizer.get_vocab_size() # updating vocab size just to capture the number we actually got after training
print("Vocabulary size:", vocabulary_size)






['I', 'adopted', 'some', 'dogs', 'from', 'the', 'animal', 'shelter']
[1, 84, 138, 33, 167, 28, 83, 183]
Vocabulary size: 195


In [8]:

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokenized_sentences.append(ids)

print("Here's what our tokenized sentences look like\n", tokenized_sentences)


Here's what our tokenized sentences look like
 [[1, 84, 138, 33, 167, 28, 83, 183], [146, 0, 19, 70, 170, 57, 33, 29, 38, 77, 150, 194], [58, 38, 54, 33, 158, 168, 83], [1, 64, 75, 57, 33, 49, 47, 175], [1, 64, 75, 57, 38, 49, 47, 190], [79, 59, 35, 28, 39], [173, 160, 165, 56, 154, 189], [33, 29, 38, 58, 192, 193, 179], [38, 29, 33, 77, 174, 166, 29, 82], [67, 31, 29, 67, 32, 130, 80], [38, 29, 33, 49, 59, 35, 28, 185, 107], [28, 177, 29, 171, 187, 84, 80], [164, 181, 35, 28, 39], [188, 29, 79, 59, 35, 28, 39], [151, 186, 191, 28, 39], [28, 39, 82, 108, 152, 29, 182], [162, 180, 58, 35, 28, 39]]


### Creating the skip grams
We'll build skip-gram pairs directly so we can see exactly what is happening.


In [9]:
import random


def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]


# Skipgrams from the first tokenized sentence
sequence_skipgrams = make_skipgrams(tokenized_sentences[0], vocabulary_size=vocabulary_size, window_size=2)
print(sequence_skipgrams)


[[(1, 84), (1, 164), (1, 138), (1, 29), (84, 1), (84, 7), (84, 138), (84, 190), (84, 33), (84, 71), (138, 1), (138, 63), (138, 84), (138, 58), (138, 33), (138, 36), (138, 167), (138, 189), (33, 84), (33, 27), (33, 138), (33, 174), (33, 167), (33, 190), (33, 28), (33, 140), (167, 138), (167, 23), (167, 33), (167, 152), (167, 28), (167, 109), (167, 83), (167, 9), (28, 33), (28, 8), (28, 167), (28, 24), (28, 83), (28, 56), (28, 183), (28, 60), (83, 167), (83, 130), (83, 28), (83, 155), (83, 183), (83, 7), (183, 28), (183, 144), (183, 83), (183, 51)], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]]


### Group Discussion

What does this output mean?

What are the different pairs of numbers?

What are all the 0s and 1s at the end?

Our `make_skipgrams` function includes a simplified version of **negative sampling**. Write a note here about what you think negative sampling is. How could we make this version better?

**Negative sampling:**



In [10]:
# let's do all of the sentences and combine the skipgrams into one big list
all_couples = []
all_labels = []
for tokenized_sentence in tokenized_sentences:
    couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size=vocabulary_size, window_size=3)
    all_couples.extend(couples)
    all_labels.extend(labels)

sequence_skipgrams = [all_couples, all_labels]

In [11]:
import torch
import torch.nn.functional as F

couples, labels = sequence_skipgrams

inputs = []
for target_word, context_word in couples:
    target_one_hot = F.one_hot(torch.tensor(target_word), num_classes=vocabulary_size)
    context_one_hot = F.one_hot(torch.tensor(context_word), num_classes=vocabulary_size)
    inputs.append(torch.cat([target_one_hot, context_one_hot]))

inputs_array = torch.stack(inputs).float()
labels_array = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print("inputs_array shape:", inputs_array.shape)
print("labels_array shape:", labels_array.shape)

print("Here's what the first input array looks like. See if you can find the two 1s for the target and context words\n", inputs_array[0])
print("Here's what the first label looks like\n", labels_array[0])


inputs_array shape: torch.Size([1056, 390])
labels_array shape: torch.Size([1056, 1])
Here's what the first input array looks like. See if you can find the two 1s for the target and context words
 tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
   

## A simple model

To build our embedding model, we need an input size of twice the vocab, because we have to input both the target and context word one-hot encodings.

Let's make it a simple fully connected network with 50 nodes in the hidden layer.

In [12]:
import torch.nn as nn
import torch.optim as optim

embedding_model = nn.Sequential(
    nn.Linear(vocabulary_size * 2, 50),
    nn.ReLU(),
    nn.Linear(50, 1)
)

loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
optimizer = optim.Adam(embedding_model.parameters(), lr=0.0001)

for epoch in range(20000):
    logits = embedding_model(inputs_array)
    loss = loss_fn(logits, labels_array)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch+1}, loss={loss.item():.4f}")


epoch 1000, loss=0.4055
epoch 2000, loss=0.2764
epoch 3000, loss=0.2147
epoch 4000, loss=0.1633
epoch 5000, loss=0.1190
epoch 6000, loss=0.0855
epoch 7000, loss=0.0634
epoch 8000, loss=0.0500
epoch 9000, loss=0.0422
epoch 10000, loss=0.0376
epoch 11000, loss=0.0350
epoch 12000, loss=0.0335
epoch 13000, loss=0.0327
epoch 14000, loss=0.0321
epoch 15000, loss=0.0318
epoch 16000, loss=0.0317
epoch 17000, loss=0.0315
epoch 18000, loss=0.0315
epoch 19000, loss=0.0314
epoch 20000, loss=0.0314


## Where is the word embedding in all this?

The weights going from the word's index to hidden layer nodes represent what the model learned about this word, so we'll use those weights as the embedding

`embedding_model[0]` is the first layer - the input layer

`.weight` is the weights we learned for that layer (a tensor)

`.detach()` stops the gradient tracking so we can look at the values without affecting the training

In [13]:
word = "dogs"
print("Word:", word)

word_ids = tokenizer.encode(word).ids # recall how BPE treats "dogs" and " dogs" as different tokens
print("token ids:", word_ids)

first_layer_weights = embedding_model[0].weight.detach() # [hidden_dim, vocab_size * 2]
word_embedding = first_layer_weights[:, word_ids[0]]        # target-word part of input

print(f"The embedding for the word '{word}' is:")
print(word_embedding)


Word: dogs
token ids: [33]
The embedding for the word 'dogs' is:
tensor([ 1.0410,  0.0563,  0.8797,  1.0368, -1.0221,  0.9096, -0.8444,  1.0312,
        -0.8744,  0.3680, -0.6354,  0.3517,  0.5154,  1.0261,  1.0858,  0.4472,
         1.0282,  1.0565, -0.3066,  0.6695,  0.5584, -0.6994,  1.0337,  1.0312,
        -0.5085,  1.0391,  0.9609, -0.6873,  0.7034,  1.0285, -0.9285,  0.8474,
         1.0327,  1.0459,  1.0897,  0.8253,  0.9764,  0.8953,  1.0082,  0.9306,
         0.1985, -0.9637, -0.9448, -0.1306,  1.0908,  0.9983,  0.7636,  0.9990,
         0.8744,  0.6695])


### We could make this into a function

In [14]:
import torch
import torch.nn.functional as F

def get_embedding(word, embedding_model):
    word_ids = tokenizer.encode(word).ids
    first_layer_weights = embedding_model[0].weight.detach()
    return first_layer_weights[:, word_ids[0]] # use first subword token for this demo


cats_embedding = get_embedding("cats", embedding_model)
dogs_embedding = get_embedding("dogs", embedding_model)
ocean_embedding = get_embedding("ocean", embedding_model)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)


tensor([-0.6747,  1.1647,  0.3307,  0.1004,  0.8968, -1.0209,  0.9756, -1.0282,
         1.1185,  0.9519, -0.2501, -0.1823,  0.9259, -0.8589,  0.4065, -0.9687,
         0.9758,  1.0645,  0.9673,  0.6499, -1.0561, -0.1284, -0.6427,  0.9414,
         0.9772,  0.3524,  0.9472, -0.9552,  0.9930,  1.0285, -0.3848, -1.0771,
         1.0264,  1.0967,  1.0280,  1.0109,  0.8768,  0.9426, -1.0063, -0.1510,
         0.1345, -0.1165,  0.9622, -0.1813, -1.0007,  0.9839, -0.5763,  0.7687,
         0.0872,  0.5945])
tensor([ 1.0410,  0.0563,  0.8797,  1.0368, -1.0221,  0.9096, -0.8444,  1.0312,
        -0.8744,  0.3680, -0.6354,  0.3517,  0.5154,  1.0261,  1.0858,  0.4472,
         1.0282,  1.0565, -0.3066,  0.6695,  0.5584, -0.6994,  1.0337,  1.0312,
        -0.5085,  1.0391,  0.9609, -0.6873,  0.7034,  1.0285, -0.9285,  0.8474,
         1.0327,  1.0459,  1.0897,  0.8253,  0.9764,  0.8953,  1.0082,  0.9306,
         0.1985, -0.9637, -0.9448, -0.1306,  1.0908,  0.9983,  0.7636,  0.9990,
         0.87

## Applied Exploration

**NOTE FROM THE FUTURE:** This exploration will likely be difficult to get working. You should use the first Applied Exploration from the next set of notes instead.

Create word embeddings for the AG News dataset.

Put all of the code into one cell so it isn't spread all throughout the notebook.

Show some example word embeddings.

Describe your results and reflect on them
* How big of a vocabulary did you need to use?
* How many training epochs do you think are appropriate? Why?
* How could you go about figuring out if these embeddings are useful?

